# Demo 06: Fine-tune EpiZoo for LoA score computation

This notebook demonstrates how to fine-tune **EpiZoo** and get **EpiZooCancer** for somatic mutation prioritization.

Starting from a pretrained EpiZoo model, users can fine-tune the model to include the cancer-type embeddings. After fine-tuning, EpiZoo can be used to compute the LoA score of each somatic mutation.

## Required files

Before running this notebook, please prepare the following files:

- **`model.pth`**: The pretrained EpiZoo model checkpoint. (The pretrained EpiZoo checkpoint can be downloaded at [pretrained_EpiZoo.pth](https://drive.google.com/file/d/1Xs5R_LAMbB_Zqpg7SFHlrAMfVcdcGwVE/view?usp=drive_link))
- **`seam.pth`**: SEAM checkpoint. (The SEAM checkpoint can be downloaded at [SEAM.pth](https://drive.google.com/file/d/1VlPnwrvMxkvKkqEOvM_pciCwJH5Jgb8Y/view?usp=drive_link))
- **`adata.h5ad`**: The scATAC-seq data. (The demo data can be downloaded at [cancer_downsampled_2000_cells.h5ad](https://drive.google.com/file/d/1KSPGLSFlVnnjB7aIFmKXHef8l2nI4no3/view?usp=drive_link))
- **`genome.fa`**: The reference genome sequence file

## Output

This notebook will generate:

- Fine-tuned EpiZooCancer model checkpoint
- LoA scores for the evaluated mutations

In [1]:
import os
import sys

# Add EpiZoo root directory
PROJECT_ROOT = os.path.abspath("../")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

## Step 1: Data processing

To prepare dataset for EpiZoo, we perform the following preprocessing steps:

1. **TF-IDF transformation:** Convert the raw count matrix into TF-IDF normalized values, which quantify the relative importance of accessible cCREs within each cell.

2. **Cell sentence generation:** Rank accessible cCREs according to their TF-IDF scores and convert each cell into a compact cell sentence composed of cCRE indices, which serves as the input for EpiZoo.

In [2]:
import scanpy as sc
import numpy as np
import pandas as pd

from epizoo.data.processing import compute_tfidf, filter_cCREs, generate_cell_sentences

# 1. Load required data files
# Load the dataset
adata_file_path = "../data/cancer_downsampled_2000_cells.h5ad"
adata = sc.read_h5ad(adata_file_path)
print(f"Anndata: {adata}")

# Load the cCRE document frequency data (for human here)
df_file_path = "../data/cCRE_frequencies_human.npy"
df = np.load(df_file_path)
print(f"Document Frequency: {df}")

# Load the cCRE filter index (for human here)
filter_index_file_path = "../data/cCRE_filter_idx_human.csv"
filter_index = pd.read_csv(filter_index_file_path,index_col=0)['idx'].values
print(f"Filter Index: {filter_index}")

# 2. Preprocess the data
# Perform TF-IDF transformation
adata = compute_tfidf(adata, df, cell_number=8200000)

# Filter cCRE
adata = filter_cCREs(adata, filter_idx=filter_index, species=0)

# Generate cell sentences
adata = generate_cell_sentences(adata, species=0)
print(f"Cell Sentences:\n{adata.obs['cell_indices'][:5]}")

/home/jiangqun/miniconda3/envs/cellemu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Anndata: AnnData object with n_obs × n_vars = 2000 × 1355445
    obs: 'BlacklistRatio', 'nDiFrags', 'nFrags', 'nMonoFrags', 'nMultiFrags', 'NucleosomeRatio', 'PassQC', 'PromoterRatio', 'ReadsInBlacklist', 'ReadsInPromoter', 'ReadsInTSS', 'Sample', 'TSSEnrichment', 'batch', 'split', 'batch_id', 'tissue'
Document Frequency: [52873.   337.  2597. ... 29708. 17642. 22866.]
Filter Index: [      0      26      27 ... 1355434 1355435 1355442]
TF-IDF completed. TF-IDF matrix stored in adata.X
Matrix shape: (2000, 1355445)
Matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Data type: float32
Non-zero entries: 34,933,448
Sparsity: 98.7114%
Non-zero value min: 0.080643
Non-zero value max: 426.961182
Non-zero value mean: 2.309390
Non-zero value median: 1.457233
--------------------------------------------------
Accessible cCREs per cell:
  Mean: 17466.72
  Median: 16237.00
  Min: 261
  Max: 57607
Filtered cCREs: 700460 features retained.
Cell Sentences:
scATAC_COAD_B5AE6F25_E2EA_412F_AC4C_E11EA47

In [3]:
from sklearn.preprocessing import LabelEncoder

# Build label mapping
label_encoder = LabelEncoder()

adata.obs["cancer_type_id"] = label_encoder.fit_transform(adata.obs["tissue"])
label_mapping = {
    cancer_type: int(idx)
    for idx, cancer_type in enumerate(label_encoder.classes_)
}

print(label_mapping)

{'BLCA': 0, 'BRCA': 1, 'COAD': 2, 'GBMx': 3, 'KIRC': 4, 'KIRP': 5, 'LUAD': 6, 'SKCM': 7}


## Step 2: Create Dataset and DataLoader

After generating cell sentences, we construct `CellDatasetCancer` and DataLoader for model fine-tuning. Each cell is represented as a sequence of tokens with special tokens [CLS] and [SEP].

In [4]:
from epizoo.data.datasets import CellDatasetCancer, collate_fn_cancer, InferenceCellDatasetCancer, inference_collate_fn_cancer
from torch.utils.data import DataLoader

# Create a dataset and dataloader for fine-tuning
human_vocab_size = 700460
mouse_vocab_size = 814020
celldataset = CellDatasetCancer(
    cell_sentences=adata.obs['cell_indices'].values,
    species=[0] * adata.n_obs,
    cancer_type=adata.obs['cancer_type_id'].values,
    cca_alpha=1,
    human_num_ccres=human_vocab_size,
    mouse_num_ccres=mouse_vocab_size,
)
dataloader = DataLoader(
    celldataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn_cancer,
)

# Create a dataset and dataloader for inference
adata_BRCA = adata[adata.obs['tissue'] == 'BRCA'].copy()
inference_dataset = InferenceCellDatasetCancer(
    cell_sentences=adata_BRCA.obs['cell_indices'].values,
    species=[0] * adata_BRCA.n_obs,
    cancer_type=adata_BRCA.obs['cancer_type_id'].values,
)
inference_dataloader = DataLoader(
    inference_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=inference_collate_fn_cancer,
)

## Step 3: Load the pretrained EpiZoo model

We load the pretrained model checkpoint for fine-tuning.

In [5]:
import torch
from epizoo.models.epizoo_cancer import EpiZooCancerConfig, EpiZooCancer

# Load pretrained model
model_path = "/data/lizhen/epizoo/models/pretrained_EpiZoo.pth"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Initialize EpiZoo model
config = EpiZooCancerConfig(
    vocab_size = human_vocab_size + mouse_vocab_size + 4,
    human_vocab_size = human_vocab_size,
    mouse_vocab_size = mouse_vocab_size,
    num_layers = 30,
    num_cancer_types = len(adata.obs['tissue'].unique()),
)
model = EpiZooCancer(cfg=config)

# Load pretrained weights
state_dict = torch.load(model_path, map_location="cpu")
msg = model.load_state_dict(state_dict, strict=False)

print("=" * 80)
print("Missing keys (new in EpiZooCancer):")
if len(msg.missing_keys) == 0:
    print("None")
else:
    for k in msg.missing_keys:
        print(f"  {k}")
print("Unexpected keys (not used in EpiZooCancer):")
if len(msg.unexpected_keys) == 0:
    print("None")
else:
    for k in msg.unexpected_keys:
        print(f"  {k}")
print("=" * 80)
print(f"Total missing keys: {len(msg.missing_keys)}")
print(f"Total unexpected keys: {len(msg.unexpected_keys)}")

# Move model to device
model = model.to(device)

print("EpiZooCancer model loaded successfully.")

Missing keys (new in EpiZooCancer):
  cca_loss_fn.pos_weight
  signal_loss_fn.pos_weight
  cancer_emb.weight
Unexpected keys (not used in EpiZooCancer):
None
Total missing keys: 3
Total unexpected keys: 0
EpiZooCancer model loaded successfully.


## Step 4: Model fine-tuning

We fine-tune the pretrained EpiZoo model on the new scATAC-seq dataset using `EpiZooFinetuneTrainer`.

The fine-tuned model checkpoint will be saved for downstream inference.

In [6]:
from epizoo.train.cancer import EpiZooCancerTrainer, CancerTrainConfig

# Fine-tuning configuration
ft_cfg = CancerTrainConfig(
    mode="sr_cca",
    output_dir="/data/lizhen/epizoo/models/finetuned_model_cancer",
    max_steps=500,
    save_steps=100,
    log_steps=50,
    keep_last=2,
    lr=5e-5,
    warmup_steps=100,
    device=device,
)

trainer = EpiZooCancerTrainer(
    model=model,
    train_loader=dataloader,
    cfg=ft_cfg,
)

# Start fine-tuning
finetuned_model = trainer.train()

print(
    "This demo demonstrates the fine-tuning workflow only. "
    "In practice, sufficient training steps are required."
)

Frozen seq_emb.
Total parameters:     2,615,697,001
Trainable parameters: 1,840,281,193
Frozen parameters:    775,415,808
Step 50 | lr=2.500e-05 | loss=1.8826 | sr=1.3013 | cca=0.5813 | cca_pos_acc=0.5850 | cca_neg_acc=0.6807 | auroc=0.6775 | auprc=0.9054
Step 100 | lr=5.000e-05 | loss=1.8567 | sr=1.2853 | cca=0.5714 | cca_pos_acc=0.7630 | cca_neg_acc=0.7961 | auroc=0.8450 | auprc=0.9586
Checkpoint saved to /data/lizhen/epizoo/models/finetuned_model_cancer/202607150822_100.pth
Step 150 | lr=5.000e-05 | loss=1.7943 | sr=1.2442 | cca=0.5501 | cca_pos_acc=0.7882 | cca_neg_acc=0.8304 | auroc=0.8709 | auprc=0.9674
Step 200 | lr=5.000e-05 | loss=1.8252 | sr=1.2793 | cca=0.5459 | cca_pos_acc=0.6248 | cca_neg_acc=0.7160 | auroc=0.7255 | auprc=0.9209
Checkpoint saved to /data/lizhen/epizoo/models/finetuned_model_cancer/202607150825_200.pth
Step 250 | lr=5.000e-05 | loss=1.8357 | sr=1.2858 | cca=0.5499 | cca_pos_acc=0.7278 | cca_neg_acc=0.8044 | auroc=0.8394 | auprc=0.9567
Step 300 | lr=5.000e-0

## Step 5: Compute LoA score for a mutation

We use the `score_mutation_loa` function to compute the LoA (Loss of Accessibility) score for a given mutation using the fine-tuned model.

In this demo, we evaluate a representative mutation located in the cCRE region **chr1:1318983-1319383**. The mutation is a single nucleotide substitution from **C to A** at position **chr1:1319024-1319025**. The reference and alternative sequences of this cCRE are generated based on the hg38 genome, followed by LoA score calculation using the fine-tuned EpiZooCancer model.

In [7]:
from epizoo.models.seam import SEAMConfig, SEAM

# Load SEAM
model_path = "/data/lizhen/epizoo/models/seam.pth"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Initialize SEAM model
config = SEAMConfig(cfg_path="../config/SEAM_config")
seam_model = SEAM(cfg=config)

# Load pretrained weights
state_dict = torch.load(model_path, map_location="cpu")

# Load SEAM
seam_model.load_state_dict(state_dict)
seam_model = seam_model.to(device)

print("SEAM model loaded successfully.")

SEAM model loaded successfully.


In [8]:
from epizoo.data.ccre import extract_dna_sequences

# Information of the evaluated mutation
# For example
mut_chr = 'chr1'
mut_start = 1319024
mut_end = 1319025
ref_base = 'C'
alt_base = 'A'
ccre_chr = 'chr1'
ccre_start = 1318983
ccre_end = 1319383
ccre_name = 'chr1:1318983-1319383'

# Get the mut_idx
mut_idx = adata_BRCA.var_names.get_loc(ccre_name)

# Get the ref and alt sequence
genome_file_path = "/data/lizhen/epizoo/genomes_fa/hg38.fa"
regions = [ccre_name]
ref_sequence = extract_dna_sequences(fasta_path=genome_file_path, regions=regions, show_progress=False)[0]
relative_pos = mut_start - ccre_start
assert ref_sequence[relative_pos].upper() == ref_base.upper(), (f"Reference mismatch: genome={ref_sequence[relative_pos]}, expected={ref_base}")
alt_sequence = (ref_sequence[:relative_pos] + alt_base + ref_sequence[relative_pos + 1:])

print("Ref sequence:", ref_sequence)
print("Alt sequence:", alt_sequence)

Ref sequence: AAGCGCTCTGAGGGCACCAGCCTGTCCCACTCTGGCCATTTCAATGCCGCTCGGACAGAGCCTGGTGGGTTCATAAGCCGGTGCATACCCCCACCCTGCACGTGCTCTCCCGGGTCGGCGCCCAGTCTGGTCTGGGACCCGCATCCTCGCGCTGACGCCTCCTGGTGTCCCCACGGTGCTGGACGCAAGAGTGAGGTGGGGTGGGCTCACCCTGAGGCCCCAGCCTTCCAGAAACTGTGTAGAACGGGCGGAGGGCATGGTTGGTGTGGGGGTGGGCAGTGACTGTGCCAGCTGTGGCCCTGGGAACCTGACCTGGACCGTCTGGTGGAGGTGGACAGCCACCACCTTCTTCATGCAGTCTTTGATCATCTGGGAGGTGAAGAAGTTGGCCTCGCCCTTC
Alt sequence: AAGCGCTCTGAGGGCACCAGCCTGTCCCACTCTGGCCATTTAAATGCCGCTCGGACAGAGCCTGGTGGGTTCATAAGCCGGTGCATACCCCCACCCTGCACGTGCTCTCCCGGGTCGGCGCCCAGTCTGGTCTGGGACCCGCATCCTCGCGCTGACGCCTCCTGGTGTCCCCACGGTGCTGGACGCAAGAGTGAGGTGGGGTGGGCTCACCCTGAGGCCCCAGCCTTCCAGAAACTGTGTAGAACGGGCGGAGGGCATGGTTGGTGTGGGGGTGGGCAGTGACTGTGCCAGCTGTGGCCCTGGGAACCTGACCTGGACCGTCTGGTGGAGGTGGACAGCCACCACCTTCTTCATGCAGTCTTTGATCATCTGGGAGGTGAAGAAGTTGGCCTCGCCCTTC


In [9]:
from epizoo.inference.mutations import score_mutation_loa

# Compute the LOA score
loa_score = score_mutation_loa(
    model=finetuned_model,
    seam_model=seam_model,
    dataloader=inference_dataloader,
    ref_sequence=ref_sequence,
    alt_sequence=alt_sequence,
    mut_idx=mut_idx,
    cfg_path='../config/SEAM_config',
    device=device,
)['score']

print(f"LoA score: {loa_score:.4f}")

Predicting signal delta: 100%|██████████| 15/15 [00:25<00:00,  1.73s/it]


LoA score: 0.0012
